# 🟡 Medium: Simple Linear Layer

Implement a fully-connected linear layer: **y = xW^T + b**

### Signature
```python
class SimpleLinear:
    def __init__(self, in_features: int, out_features: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- `self.weight`: shape `(out_features, in_features)`, init with `randn * (1/√in_features)`
- `self.bias`: shape `(out_features,)`, init as zeros
- Both must have `requires_grad=True`
- `forward(x)` computes `x @ W^T + b`
- Do **NOT** use `torch.nn.Linear`

In [1]:
import torch
import math

In [4]:
# ✏️ YOUR IMPLEMENTATION HERE

class SimpleLinear:
    def __init__(self, in_features: int, out_features: int):

        #weight要求是randn*sqrt(in_feature)
        self.weight = torch.randn(out_features,in_features)*(1/math.sqrt(in_features))
        self.weight.requires_grad_(True)#grad_是原地操作，一般用于初始化
        self.bias = torch.zeros(out_features,requires_grad = True)

          # Initialize weight and bias

    def forward(self, x: torch.Tensor) -> torch.Tensor:#这里的x一般是batch，in_features的维度，所以w要转置一下
        return x @ self.weight.T + self.bias
          # Compute y = x @ W^T + b

在深度学习与神经网络架构设计中，将标准正态分布生成的权重乘以 1 / math.sqrt(in_features) 是一种标准的方差缩放初始化策略（类似于 LeCun 初始化或简化版的 Xavier 初始化）。其核心工程目的在于保持网络在深层传递时数据方差的稳定性，防范梯度消失或梯度爆炸现象。可以从以下严谨的数学推导与算法工程维度进行解析：

1. 维持前向传播的特征方差恒定线性层输出 $Y$ 的第 $i$ 个元素的数学定义为两向量的点积：

$$y_i = \sum_{j=1}^{in\_features} W_{i,j} x_j$$

假设输入特征 $x$ 与权重 $W$ 的元素均服从均值为 0 的独立同分布。根据概率论方差的性质，输出 $y_i$ 的方差可推导为：

$$Var(y_i) = \sum_{j=1}^{in\_features} Var(W_{i,j} \cdot x_j) = in\_features \cdot Var(W) \cdot Var(x)$$

若权重 $W$ 仅由 torch.randn 生成（其标准差为 1，方差 $Var(W) = 1$），代入上式可知：

$$Var(y) = in\_features \cdot Var(x)$$

这表明，输入张量每经过一层未经缩放的线性映射，其数据的方差就会被放大 in_features 倍。在深层网络模型中，这种逐层累乘的放大效应会导致网络输出的数值迅速超出浮点数表示范围（发生溢出），并使得数值落入后续激活函数的饱和区，导致梯度计算失效。


为了确保信号在网络中传播时保持稳定，工程上的最优解是令该层的输出方差等于输入方差，即期望 $Var(y) = Var(x)$。将此目标代入前述方程，必须满足：

$$in\_features \cdot Var(W) = 1$$

由此解得，权重矩阵合理的方差应为：
$$Var(W) = \frac{1}{in\_features}$$

由于正态分布的缩放是通过标准差（方差的平方根）实现的，因此需要将方差为 1 的标准正态分布乘以以下常数：

$$\sigma = \frac{1}{\sqrt{in\_features}}$$

这就是代码中乘以 1/math.sqrt(in_features) 的根本原因。它确保了初始化后的权重矩阵 $W$ 严格服从 $N(0, \frac{1}{in\_features})$ 的分布。

In [5]:
# 🧪 Debug
layer = SimpleLinear(8, 4)
print("W shape:", layer.weight.shape)  # should be (4, 8)
print("b shape:", layer.bias.shape)    # should be (4,)

x = torch.randn(2, 8)
y = layer.forward(x)
print("Output shape:", y.shape)        # should be (2, 4)

W shape: torch.Size([4, 8])
b shape: torch.Size([4])
Output shape: torch.Size([2, 4])


In [6]:
# ✅ SUBMIT
from torch_judge import check
check("linear")


🧪 Testing: Simple Linear Layer (Medium)
──────────────────────────────────────────────────
  ✅ [1/3] Weight & bias shape (0.3ms)
  ✅ [2/3] Forward pass (0.4ms)
  ✅ [3/3] Gradient flow (30.6ms)
──────────────────────────────────────────────────
  🎉 All 3 tests passed! (31.3ms total)
  Progress saved. Run status() to see your dashboard.

